# Week 10 Practice: RNNs and LSTMs

CSCI1409 Deep Learning · based on the RNN and LSTM lecture

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Do not use AI tools for any task (ChatGPT, Claude, Gemini, Copilot, Google AI Overviews).
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.

In [2]:
import torch
import torch.nn as nn

## Task 1: The RNN cell

An RNN reads a sequence one element at a time. After each element it updates its **hidden state** $h_t$, a vector that summarizes everything it has read so far:

$$h_t = \tanh\left(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b\right), \qquad h_0 = 0$$

The same $W_{xh}$, $W_{hh}$ and $b$ are used at every step.

### 1a) A tiny RNN by hand

Use an RNN where everything is a single number: $w_x = 1$, $w_h = 0.5$, $b = 0$, $h_0 = 0$, so

$$h_t = \tanh(1 \cdot x_t + 0.5 \cdot h_{t-1})$$

1. Compute $h_1, h_2, h_3$ on paper for the sequence $A = [1, 0, 0]$. Write the calculation for each step. You may use `torch.tanh` or a calculator for the tanh values.
2. Write a `for` loop in PyTorch that computes all hidden states. Run it on $A$ and on $B = [0, 0, 1]$.
3. $A$ and $B$ contain the same numbers. Why is $h_3$ different? What happens to the "1" from step 1 of $A$ as time goes on?

In [7]:
w_x, w_h, b = 1.0, 0.5, 0.0

A = torch.tensor([1., 0., 0.])
B = torch.tensor([0., 0., 1.])

# Your code here
full_A = len(A)
h_A = [torch.tensor(0.0)]
for i in range(full_A):
    h_next = torch.tanh(w_x * A[i] + w_h * h_A[i] + b)
    h_A.append(h_next)

full_B = len(B)
h_B = [torch.tensor(0.0)]
for i in range(full_B):
    h_next = torch.tanh(w_x * B[i] + w_h * h_B[i] + b)
    h_B.append(h_next)
    
print("A:", [h.item() for h in h_A[1:]])
print("B:", [h.item() for h in h_B[1:]])

A: [0.7615941762924194, 0.3633995056152344, 0.17972621321678162]
B: [0.0, 0.0, 0.7615941762924194]


**Your answer:**

### 1b) `nn.RNN`: shapes and parameters

With input size $D$ and hidden size $H$, an `nn.RNN` layer has

$$\text{params} = H \cdot D + H \cdot H + 2H$$

($W_{xh}$ is $H \times D$, $W_{hh}$ is $H \times H$, and PyTorch keeps two bias vectors of size $H$.)

With `batch_first=True`, the input has shape `[batch, T, D]`. The layer returns two things: `out` (the hidden state at every step) and `h_n` (the last hidden state).

| Case | $D$ | $H$ | batch | $T$ |
|---|---|---|---|---|
| A | 1 | 4 | 2 | 5 |
| B | 8 | 16 | 4 | 10 |
| C | 8 | 16 | 4 | 100 |
| D | 3 | 32 | 1 | 7 |

1. Predict the shape of `out`, the shape of `h_n` and the number of parameters for each case.
2. Check with `nn.RNN(D, H, batch_first=True)`. Pass `torch.randn(batch, T, D)` through it and count parameters with `sum(p.numel() for p in rnn.parameters())`.
3. B and C have the same number of parameters, but C reads a sequence 10 times longer. Why does the count not change?

In [12]:
# Your code here
A = [2, 5, 4, 1]
B = [4, 10, 16, 8]
C = [4, 100, 16, 8]
D = [1, 7, 32, 3]

def mod(case):
    rnn = nn.RNN(case[3], case[2], batch_first=True)
    x = torch.randn(case[0], case[1], case[3])    
    out, h_n = rnn(x)
    params = sum(p.numel() for p in rnn.parameters())
    
    return out.shape, h_n.shape, params

for name, case in [("A", A), ("B", B), ("C", C), ("D", D)]:
    out_shape, h_n_shape, params = mod(case)
    print(name, "out:", out_shape, "h_n:", h_n_shape, "params:", params)

A out: torch.Size([2, 5, 4]) h_n: torch.Size([1, 2, 4]) params: 28
B out: torch.Size([4, 10, 16]) h_n: torch.Size([1, 4, 16]) params: 416
C out: torch.Size([4, 100, 16]) h_n: torch.Size([1, 4, 16]) params: 416
D out: torch.Size([1, 7, 32]) h_n: torch.Size([1, 1, 32]) params: 1184


**Your answer:**
B and C have the same number of parameters because both RNNs have the same input size (D = 8) and hidden size (H = 16). The RNN reuses the same weights at every time step. Case C applies those weights 100 times instead of 10, so it requires more computation but no additional parameters.

## Task 2: LSTM

An LSTM keeps two memories: the hidden state $h_t$ and the **cell state** $c_t$ (long-term memory). Three **gates** with values between 0 and 1 (from a sigmoid) control the cell state, and a candidate $g_t$ (between -1 and 1, from tanh) is the new information:

- forget gate $f_t$: how much of the old memory to keep,
- input gate $i_t$: how much of the new information to write,
- output gate $o_t$: how much of the memory to show as $h_t$.

$$c_t = f_t \cdot c_{t-1} + i_t \cdot g_t, \qquad h_t = o_t \cdot \tanh(c_t)$$

### 2a) One LSTM step by hand

The old cell state is $c_{t-1} = 2$. The gates are already computed:

| Case | $f_t$ | $i_t$ | $g_t$ | $o_t$ |
|---|---|---|---|---|
| A | 1 | 0 | 0.5 | 1 |
| B | 0 | 1 | 0.5 | 1 |
| C | 0.5 | 0.5 | -1 | 0.5 |
| D | 0.9 | 0.2 | 0.5 | 0.8 |

1. Compute $c_t$ and $h_t$ on paper for each case.
2. Check with PyTorch. Put each column in a tensor and compute all four cases at once.
3. In which case does the LSTM keep its memory exactly? In which case does it throw the old memory away and write new information? What would $o_t = 0$ mean?

In [25]:
c_prev = 2.0

# Your code here
def lstm(f_t, c_prev, i_t, g_t, o_t):
    c_t = f_t * c_prev + i_t * g_t
    h_t = o_t * torch.tanh(c_t)
    return c_t, h_t

A = torch.tensor([1,0,0.5,1])
B = torch.tensor([0,1,0.5,1])
C = torch.tensor([0.5,0.5,-1,0.5])
D = torch.tensor([0.9,0.2,0.5,0.8])

cases = torch.stack([A, B, C, D])
print(cases.shape)

c_t, h_t = lstm(cases[:, 0], c_prev,cases[:, 1], cases[:, 2], cases[:, 3])

print("c_t:", c_t)
print("h_t:", h_t)

torch.Size([4, 4])
c_t: tensor([2.0000, 0.5000, 0.5000, 1.9000])
h_t: tensor([0.9640, 0.4621, 0.2311, 0.7650])


**Your answer:**

### 2b) An LSTM for sequence classification

A common use: read a whole sequence and output one class (for example, the sentiment of a sentence). This is called **many-to-one**: we use only the last hidden state.

```python
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)
```

1. Pass `x = torch.randn(4, 10, 8)` through `lstm`. It returns `out, (h_n, c_n)`. Print the three shapes.
2. Take the last time step with `out[:, -1, :]` and pass it through `fc`. What is the shape of the result and what does it mean? Check with `torch.allclose` that `out[:, -1, :]` equals `h_n[0]`.
3. Count the parameters of `lstm` and of `nn.RNN(8, 16)`. Why does the LSTM have exactly 4 times more?

In [24]:
# Your code here
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)

x = torch.randn(4,10,8)
out, (h_n, c_n) = lstm(x)
print("out:", out.shape)
print("h_n:", h_n.shape)
print("c_n:", c_n.shape)

last_time = out[:, -1, :]
log = fc(last_time)

print("last hidden:", last_time.shape)
print("logits:", log.shape)
print("Same as h_n[0]:", torch.allclose(last_time, h_n[0]))

rnn = nn.RNN(input_size=8, hidden_size=16, batch_first=True)

lstm_params = sum(p.numel() for p in lstm.parameters())
rnn_params = sum(p.numel() for p in rnn.parameters())

print("LSTM parameters:", lstm_params)
print("RNN parameters:", rnn_params)
assert lstm_params/rnn_params == 4

out: torch.Size([4, 10, 16])
h_n: torch.Size([1, 4, 16])
c_n: torch.Size([1, 4, 16])
last hidden: torch.Size([4, 16])
logits: torch.Size([4, 3])
Same as h_n[0]: True
LSTM parameters: 1664
RNN parameters: 416


**Your answer:**

## Stretch (optional): many-to-many

Some tasks need an output at **every** step, for example a tag for every word in a sentence. Apply `fc` to the whole `out` from 2b (not only the last step). What is the shape of the result, and what does each part mean?

In [ ]:
# Your code here